# Obstacle Detection - Seven-Class YOLOv8n Training

This corrected notebook validates a user-provided YOLO dataset before training. Required class order: `person`, `vehicle`, `animal`, `rock`, `stump`, `fence`, `ditch`.

Upload one ZIP containing `dataset.yaml` and `train/`, `valid/`, and `test/` folders. The notebook rejects missing labels, malformed boxes, empty train/valid splits, and absent classes before GPU training starts.

**Outputs:** `obstacle_model_quant.tflite`, `obstacle_model_quant_edgetpu.tflite`, `obstacle_labels.txt`, and `best.pt`.


In [ ]:
# Install current dependencies. Restart the runtime only if Colab asks.
%pip install -q -U "ultralytics>=8.4.83" kagglehub pyyaml matplotlib


In [ ]:
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from IPython.display import Image, display
from ultralytics import YOLO

assert torch.cuda.is_available(), "GPU not enabled: Runtime > Change runtime type > T4 GPU"
DEVICE = 0
IMGSZ_TRAIN = 640
IMGSZ_EXPORT = 320
SEED = 42
random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))


## Required ZIP layout

```text
obstacle_dataset.zip
  dataset.yaml
  train/images/...
  train/labels/...
  valid/images/...
  valid/labels/...
  test/images/...
  test/labels/...
```

Each label line must be: `<class_id> <x_center> <y_center> <width> <height>`, normalized to 0..1.


In [ ]:
# Upload the prepared obstacle_dataset.zip (not thousands of files one by one).
from google.colab import files
import zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert len(zip_names) == 1, f"Upload exactly one ZIP; received {list(uploaded)}"
EXTRACT_ROOT = Path("/content/obstacle_upload")
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(EXTRACT_ROOT)
print("Extracted to:", EXTRACT_ROOT)


In [ ]:
# Locate dataset.yaml even if the ZIP contains one outer folder.
yaml_candidates = list(EXTRACT_ROOT.rglob("dataset.yaml")) + list(EXTRACT_ROOT.rglob("data.yaml"))
assert len(yaml_candidates) == 1, f"Expected one dataset YAML, found {yaml_candidates}"
SOURCE_YAML = yaml_candidates[0]
DATASET_DIR = SOURCE_YAML.parent.resolve()
CLASS_NAMES = ["person", "vehicle", "animal", "rock", "stump", "fence", "ditch"]

source_config = yaml.safe_load(SOURCE_YAML.read_text()) or {}
source_names = source_config.get("names")
if isinstance(source_names, dict):
    source_names = [
        source_names[index] if index in source_names else source_names[str(index)]
        for index in range(len(source_names))
    ]
assert source_names == CLASS_NAMES, (
    f"Class order must be {CLASS_NAMES}; found {source_names}"
)

DATASET_CONFIG = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
YAML_PATH = Path("/content/obstacle_dataset.yaml")
YAML_PATH.write_text(
    yaml.safe_dump(DATASET_CONFIG, sort_keys=False), encoding="utf-8"
)
print(YAML_PATH.read_text())


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def validate_yolo_dataset(dataset_dir, class_names):
    dataset_dir = Path(dataset_dir)
    errors = []
    counts = {split: 0 for split in ("train", "valid", "test")}
    class_counts = [0] * len(class_names)

    for split in counts:
        image_dir = dataset_dir / split / "images"
        label_dir = dataset_dir / split / "labels"
        if not image_dir.is_dir() or not label_dir.is_dir():
            errors.append(f"Missing {image_dir} or {label_dir}")
            continue

        images = sorted(
            path for path in image_dir.iterdir()
            if path.suffix.lower() in IMAGE_EXTENSIONS
        )
        counts[split] = len(images)
        for image_path in images:
            label_path = label_dir / f"{image_path.stem}.txt"
            if not label_path.exists():
                errors.append(f"Missing label: {label_path}")
                continue

            for line_no, line in enumerate(label_path.read_text().splitlines(), 1):
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    errors.append(f"{label_path}:{line_no}: expected 5 values")
                    continue
                try:
                    class_id = int(parts[0])
                    coords = [float(value) for value in parts[1:]]
                except ValueError:
                    errors.append(f"{label_path}:{line_no}: non-numeric value")
                    continue

                if not 0 <= class_id < len(class_names):
                    errors.append(f"{label_path}:{line_no}: class {class_id} out of range")
                elif all(0.0 <= value <= 1.0 for value in coords) and coords[2] > 0 and coords[3] > 0:
                    class_counts[class_id] += 1
                else:
                    errors.append(f"{label_path}:{line_no}: invalid normalized box {coords}")

    if errors:
        print("First dataset errors:")
        print("\n".join(errors[:30]))
        raise ValueError(f"Dataset validation failed with {len(errors)} error(s)")
    if counts["train"] == 0 or counts["valid"] == 0:
        raise ValueError(f"Train and valid splits must be non-empty: {counts}")

    missing_classes = [
        name for name, count in zip(class_names, class_counts) if count == 0
    ]
    if missing_classes:
        raise ValueError(f"No labeled objects found for classes: {missing_classes}")

    print("Images by split:", counts)
    print("Objects by class:", dict(zip(class_names, class_counts)))

validate_yolo_dataset(DATASET_DIR, CLASS_NAMES)


In [ ]:
# Use YOLOv8n for a stable raw detection head compatible with pi/ai/yolo_tflite.py.
# Do not switch to an end-to-end/NMS-embedded export unless the Pi decoder is updated.
model = YOLO("yolov8n.pt")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")


In [ ]:
results = model.train(
    data=str(YAML_PATH),
    epochs=150,
    imgsz=IMGSZ_TRAIN,
    batch=16,
    patience=25,
    device=DEVICE,
    workers=2,
    seed=SEED,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    project="obstacle_detection",
    name="yolov8n_obstacles",
    plots=True,
)
BEST_PT = Path(results.save_dir) / "weights" / "best.pt"
assert BEST_PT.exists(), BEST_PT
print("Best model:", BEST_PT)


In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=str(YAML_PATH), imgsz=IMGSZ_TRAIN, device=DEVICE)
print(
    f"mAP50={metrics.box.map50:.4f}  mAP50-95={metrics.box.map:.4f}  "
    f"precision={metrics.box.mp:.4f}  recall={metrics.box.mr:.4f}"
)
for class_id, name in enumerate(CLASS_NAMES):
    if class_id < len(metrics.box.ap50):
        print(f"{name:10s}: AP50={metrics.box.ap50[class_id]:.4f}")
for plot_name in ("results.png", "confusion_matrix_normalized.png", "PR_curve.png"):
    plot_path = Path(results.save_dir) / plot_name
    if plot_path.exists():
        display(Image(filename=str(plot_path), width=800))


In [ ]:
MODEL_CPU_NAME = "obstacle_model_quant.tflite"
MODEL_EDGE_NAME = "obstacle_model_quant_edgetpu.tflite"
LABELS_FILE = Path("obstacle_labels.txt")

# One Edge TPU export creates both required deployment files:
#   *_full_integer_quant.tflite          (CPU INT8 fallback)
#   *_full_integer_quant_edgetpu.tflite  (Coral-compiled)
# This legacy TensorFlow export keeps NHWC input and the raw YOLO head expected
# by pi/ai/yolo_tflite.py. Do not replace the CPU file with format="litert":
# current LiteRT exports use NCHW input and need a different Pi preprocessor.
best_model = YOLO(str(BEST_PT))

edge_export = Path(best_model.export(
    format="edgetpu",
    imgsz=IMGSZ_EXPORT,
    quantize=8,
    data=str(YAML_PATH),
    fraction=1.0,
))
source_int8 = edge_export.with_name(
    edge_export.name.replace("_edgetpu.tflite", ".tflite")
)
assert source_int8.exists(), f"Uncompiled INT8 model not found beside {edge_export}"

CPU_MODEL = Path.cwd() / MODEL_CPU_NAME
EDGE_MODEL = Path.cwd() / MODEL_EDGE_NAME
shutil.copy2(source_int8, CPU_MODEL)
shutil.copy2(edge_export, EDGE_MODEL)

print(f"CPU model: {CPU_MODEL} ({CPU_MODEL.stat().st_size / 1024 / 1024:.2f} MB)")
print(f"Edge TPU model: {EDGE_MODEL} ({EDGE_MODEL.stat().st_size / 1024 / 1024:.2f} MB)")


In [ ]:
# Verify the CPU model's tensor contract against pi/ai/yolo_tflite.py.
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite import Interpreter

interpreter = Interpreter(model_path=str(CPU_MODEL))
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]
input_shape = tuple(int(value) for value in input_detail["shape"])
output_shape = tuple(int(value) for value in output_detail["shape"])
expected_channels = 4 + len(CLASS_NAMES)

assert input_shape == (1, IMGSZ_EXPORT, IMGSZ_EXPORT, 3), (
    "Pi preprocessing expects NHWC input; got", input_shape
)
assert expected_channels in output_shape, (
    f"Pi decoder expects a raw YOLO channel dimension of {expected_channels}; got {output_shape}"
)
assert EDGE_MODEL.exists() and EDGE_MODEL.stat().st_size > 0, EDGE_MODEL

expected_names = {index: name for index, name in enumerate(CLASS_NAMES)}
assert best_model.names == expected_names, (best_model.names, expected_names)
LABELS_FILE.write_text("\n".join(CLASS_NAMES) + "\n", encoding="utf-8")
print("Input:", input_shape, input_detail["dtype"], input_detail["quantization"])
print("Output:", output_shape, output_detail["dtype"], output_detail["quantization"])
print("Labels written to:", LABELS_FILE)


In [ ]:
from google.colab import files

for artifact in (CPU_MODEL, EDGE_MODEL, LABELS_FILE, BEST_PT):
    files.download(str(artifact))

print("Download complete. Keep the exact deployment filenames shown above.")


## Deploy and verify on the Raspberry Pi

1. Copy `obstacle_model_quant_edgetpu.tflite`, `obstacle_model_quant.tflite`, and `obstacle_labels.txt` into `models/`.
2. Keep the seven labels in exactly the notebook order.
3. From the repository root run `python3 pi/ai/benchmark.py --image path/to/obstacle.jpg --iters 50`.
4. Confirm `ObstacleDetector: TFLite backend ready` and physically test STOP behavior with motors lifted off the ground or drive power disconnected.
5. Do not rely on a Colab/PyTorch FPS estimate; only the Raspberry Pi + Coral benchmark is meaningful for rover safety.


## Optional: quantization-cost check and QAT notes

Everything above is the normal pipeline — this section is an **optional add-on** that starts from the existing `best.pt` and never changes the deployment filenames.

**Calibration status:** the INT8 export above is post-training quantization (PTQ) and is already calibrated as well as the ultralytics exporter allows — `data=` points at the full dataset YAML and `fraction=1.0` runs **every** training image through calibration (the default calibrates on a subset). There is no extra calibration knob left to turn here.

**Why no QAT cell:** ultralytics has **no first-class quantization-aware-training API**, so this notebook does not pretend to have one. What it *can* do honestly is **measure** the INT8 cost, so you know whether QAT would even pay off: the cells below validate the FP32 `best.pt` and the deployed INT8 `obstacle_model_quant.tflite` on the same validation split at the 320 px deployment resolution and print the mAP difference. For this safety-critical model, treat any meaningful drop on the `person` class as a blocker regardless of the overall mAP.

**What true QAT would require** (only worth it if the measured drop is large, e.g. more than ~2–3 mAP50 points): a custom PyTorch QAT loop around the YOLO model (`torch.ao.quantization` FX graph mode with fake-quant observers, then re-running the ultralytics trainer on the prepared model), or a third-party toolchain such as Intel Neural Compressor or NVIDIA's TensorRT-oriented quantization toolkits. None of these export directly to the NHWC raw-head TFLite file that `pi/ai/yolo_tflite.py` expects, so they would also need a custom export path — out of scope for this notebook.

**How to run:** in the same session right after training + export, or in a fresh session after re-running the install, imports, dataset-upload, and export cells. The resolver cell below also accepts an uploaded `best.pt`; after resolving it you can re-run the existing export cell to regenerate the same output files.

In [ ]:
# Resolve the checkpoint for the quantization-cost check. Priority:
#   1. BEST_PT from the training cell earlier in this session
#   2. newest obstacle_detection/*/weights/best.pt already on this runtime
#   3. manual upload of a previously downloaded best.pt
try:
    BEST_PT
except NameError:
    BEST_PT = None
if BEST_PT is None or not Path(BEST_PT).exists():
    candidates = sorted(
        Path("obstacle_detection").glob("*/weights/best.pt"),
        key=lambda path: path.stat().st_mtime,
    )
    if candidates:
        BEST_PT = candidates[-1]
    else:
        from google.colab import files
        print("No best.pt on this runtime - upload the one you downloaded "
              "after training:")
        uploaded = files.upload()
        names = [name for name in uploaded if name.endswith(".pt")]
        assert names, "Upload exactly one .pt checkpoint"
        BEST_PT = Path(names[0]).resolve()
BEST_PT = Path(BEST_PT)
print("Checkpoint:", BEST_PT)
# If obstacle_model_quant.tflite / obstacle_model_quant_edgetpu.tflite are
# missing in this session, re-run the EXISTING export cell above now (do not
# duplicate it): it reads BEST_PT and rewrites the same deployment filenames.

In [ ]:
# Measure the INT8 quantization cost: FP32 best.pt vs the deployed INT8
# TFLite, both validated on the SAME val split at the 320 px deployment
# resolution. This is the number that tells you whether QAT is even needed.
assert "YAML_PATH" in globals() and Path(YAML_PATH).exists(), (
    "dataset YAML missing - re-run the dataset upload/validation cells above"
)
cpu_model_path = Path.cwd() / "obstacle_model_quant.tflite"
assert cpu_model_path.exists(), (
    f"{cpu_model_path} not found - re-run the export cell above first"
)

fp32_metrics = YOLO(str(BEST_PT)).val(
    data=str(YAML_PATH), imgsz=IMGSZ_EXPORT, device=DEVICE, verbose=False
)
# TFLite inference runs on CPU with batch 1 (the model has a fixed batch dim).
int8_metrics = YOLO(str(cpu_model_path), task="detect").val(
    data=str(YAML_PATH), imgsz=IMGSZ_EXPORT, device="cpu", batch=1,
    verbose=False,
)

print("\n=== Quantization cost on the val split @ 320 px ===")
print(f"FP32 best.pt              : mAP50={fp32_metrics.box.map50:.4f}  "
      f"mAP50-95={fp32_metrics.box.map:.4f}")
print(f"INT8 {cpu_model_path.name}: mAP50={int8_metrics.box.map50:.4f}  "
      f"mAP50-95={int8_metrics.box.map:.4f}")
print(f"mAP50 lost to quantization: "
      f"{fp32_metrics.box.map50 - int8_metrics.box.map50:+.4f}")
print("Rule of thumb: a drop under ~0.02-0.03 mAP50 is normal for INT8 PTQ; "
      "anything larger, see the QAT notes above. For a safety model, also "
      "compare the per-class 'person' AP between the two runs.")